In [1]:
#用户提出问题->agent接收->求出大模型->告知agent用哪个工具->将agent返回的结果以一定样式输出
import json
import os
from openai import OpenAI
from dotenv import load_dotenv

###第一步,程序本身要有工具
#1.===================定义工具(函数)===================
##定义天气函数
def get_weather(location: str) -> str:
    """查看天气用此工具"""
    return f"It's sunny today in {location}!"


##定义一个计算的工具
def calculator(x: float) -> float:
    """计算平方根,用此工具"""
    return x ** 0.5

###第二步,有工具,但大模型不知道如何使用,需要详细描述每个工具的参数信息,如函数名,参数列表,参数类型,参数描述等
###这些信息将帮助大模型理解如何使用这些工具,tools在第一次会话大模型时,会作为参数传递给大模型
#2.=============定义tools列表,详细描述每个工具的参数信息=============
# 把函数放到dict中，方便根据函数名查找
tools_by_name = {"get_weather": get_weather, "calculator": calculator}
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather information for a specific location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The location to check the weather for"
                    }
                },
                "required": [
                    "location"
                ]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "A simple calculator tool",
            "parameters": {
                "type": "object",
                "properties": {
                    "x": {
                        "type": "number",
                        "description": "The number to calculate the square root of"
                    }
                },
                "required": [
                    "x"
                ],
                "strict": "true"
            }
        }
    }
]


##第三步,定义LLM客户端,即大模型,从env中读取api和url连接地址(即程序与大模型的连接,传递参数的通道)
#3.=============定义LLM客户端,即大模型=============
client = OpenAI(
    api_key="sk-8fc6bb5c387447b4a80dc9517678b265",
    base_url="https://api.deepseek.com/v1"
)

##如何将问题传递给大模型,如何接收大模型返回的信息
#4.=============定义获取信息的方法=============
def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
        tools=tools,
        temperature=0,
        extra_body={"thinking": {"type": "disabled"}}
    )
    return response.choices[0].message

# #要传递给大模型的问题,模拟用户输入
# messages=[
#             {"role":"system", "content":"你是一个AI高级开发工程师"},
#             {"role":"user","content":"请讲解下什么是python？字数在100字内"}
#         ]
# #调用函数,将问题传递给大模型,并接收大模型返回的信息
# #大模型返回的信息,包括大模型提出要哪个工具,且此工具的实际参数等信息
# response = send_messages(messages)
# #输出大模型返回的信息
# print(response)



##第五步,定义执行Tool调用的方法,这一步是将大模型返回的Tool信息进行解析,并调用相应的工具函数得到的结果返回给大模型
# ==================== 5. 定义执行Tool调用的方法 ====================
#message：大模型此次返回的信息->主要是tool_call字段,包含要调用的工具名和实际参数
#messages：大模型与用户往来的所有信息->主要是为了将Tool结果追加到消息列表中

def execute_tool(message, messages):
    """解析Tool信息，执行对应的函数，并将结果追加到消息列表"""
    tool_call = message.tool_calls[0]
    tool_id = tool_call.id
    tool_name = tool_call.function.name#工具名
    tool_args = json.loads(tool_call.function.arguments)#工具参数

    print(f"调用工具: {tool_name}, 参数: {tool_args}")

    # 根据函数名找到对应函数并执行
    result = tools_by_name[tool_name](**tool_args)
    print(f"工具结果: {result}")

    # 将AI消息追加到消息列表
    messages.append(message)
    # 将本次程序调用结果追加到消息列表中
    messages.append({
        "role": "tool",
        "tool_call_id": tool_id,
        "content": result
    })
    # 返回更新后的消息列表
    return messages


##所有准备工作完成,主流程模拟开始
# ==================== 6. 主流程 ====================
# 准备用户消息
messages = [{"role": "user", "content": "杭州天气怎么样?"}]
print(f"User> {messages[0]['content']}")

# 第一次调用LLM，获取ToolCall
message = send_messages(messages)
print(f"Tool> {message.tool_calls[0]}")

# 执行Tool并将结果传回LLM
messages = execute_tool(message, messages)

# 再次调用LLM，生成最终答案
message = send_messages(messages)
print(f"AI> {message.content}")

User> 杭州天气怎么样?
Tool> ChatCompletionMessageFunctionToolCall(id='call_00_tzxHRiczRJeUSCp4HP9S4913', function=Function(arguments='{"location": "杭州"}', name='get_weather'), type='function', index=0)
调用工具: get_weather, 参数: {'location': '杭州'}
工具结果: It's sunny today in 杭州!
AI> 杭州今天天气晴朗，阳光明媚！☀️ 是个适合外出活动的好天气。如果你有计划出门，记得做好防晒措施哦。需要了解更多天气信息或其他帮助吗？
